# Data Cleaning with Pandas

In this notebook we'll go through a few basic data cleaning steps that should be performed on all new datasets where necessary.

We'll go through the process with both the `orders` and `orderlines` datasets. You can then practice these skills by cleaning the `products` dataset yourself

In [1]:
import pandas as pd

In [4]:
## orders.csv
# url = "https://drive.google.com/file/d/1Vu0q91qZw6lqhIqbjoXYvYAQTmVHh6uZ/view?usp=drive_link"
# path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders = pd.read_csv(r"C:\Users\msnch\OneDrive\Bootcamp\Eniacs_Discount\1_original\orders.csv")

## orderlines.csv
# url = "https://drive.google.com/file/d/1FYhN_2AzTBFuWcfHaRuKcuCE6CWXsWtG/view?usp=drive_link"
# path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(r"C:\Users\msnch\OneDrive\Bootcamp\Eniacs_Discount\1_original\orderlines.csv")

Before we begin, let's create a copy of the `orders` and `orderlines` DataFrames. This way we are sure any of our changes won't affect the original DataFrames

In [5]:
orders_df = orders.copy()

In [5]:
orderlines_df = orderlines.copy()

One of the best ways to begin data cleaning is by exploring using `.info()`. This will tell us:
* The shape of the DataFrame
* The names of the columns
* If there are any missing values
* The datatypes of the columns

By exploring the missing values and correcting any incorrect datatypes, we often come across inconsistencies in our data.

Beyond this, we should also have a **check for any duplicate rows**.

Let's first deal with the duplicates, as it's nice and easy, then we'll explore what `.info()` has to tell us.

## 1.&nbsp; Duplicates
We can check for duplicates using the pandas [.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) method.

We can then delete these rows, if we wish, using [.drop_duplicates()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

In [6]:
# orders_df
orders_df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  str    
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 13.7 MB


In [7]:
# orderlines_df
orderlines_df.duplicated().sum()

np.int64(0)

We have no duplicate rows in either DataFrame. Easy, there is no problem to solve. Normally though, if there were some duplicates, we'd drop the extra rows.

# 2.&nbsp; `.info()`

In [8]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  str    
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 13.7 MB


* `total_paid` has 5 missing values
* `created_date` should become datetime datatype

In [10]:
orderlines_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                293983 non-null  int64
 1   id_order          293983 non-null  int64
 2   product_id        293983 non-null  int64
 3   product_quantity  293983 non-null  int64
 4   sku               293983 non-null  str  
 5   unit_price        293983 non-null  str  
 6   date              293983 non-null  str  
dtypes: int64(4), str(3)
memory usage: 24.6 MB


* `date` should be a datetime datatype
* `unit_price` should be a float datatype

## 3.&nbsp; Missing values

### 3.1.&nbsp; Orders
* `total_paid` has 5 missing values

In [ ]:
num_missing = orders_df['total_paid'].isna().sum()
total_rows = orders_df.shape[0]
percent_missing = (100*num_missing/total_rows)
print(f"5 missing values represents {percent_missing:.5f}% of the rows in our DataFrame")

5 missing values represents 0.00220% of the rows in our DataFrame


> A quick way to find out a percentage if you don't need to print out a sentence for yourself/students/colleagues is `.value_counts(normalize=True)`

In [ ]:
orders_df['total_paid'].isna().value_counts(normalize=True)

,proportion
total_paid,
False,0.999978
True,0.000022


As there is such a tiny amount of missing values, we will simply delete these rows, as we have enough data without them.

In [ ]:
orders_df = orders_df.dropna(axis=0) # Why does this drops all missing values?

Should you have a significant number of missing values in the future, you have a choice:
+ you can impute the values
+ you can take the values from other DataFrames if they are redundantly stored
+ you can delete the rows or columns
+ or any number of other creative solutions

Please, always consider how much time you have on your project, and what impact your method of choice will have on your final assesment.

### 3.2.&nbsp; Orderlines
There are no missing values in `orderlines_df`

## 4.&nbsp; Datatypes

### 4.1.&nbsp; Orders
* `created_date` should become datetime datatype

In [ ]:
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

### 4.1.&nbsp; Orderlines
* `date` should be a datetime datatype
* `unit_price` should be a float datatype

#### 4.1.1.&nbsp; `date`

In [ ]:
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

#### 4.1.2.&nbsp;`unit_price`

In [ ]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

ValueError: Unable to parse string "1.137.99" at position 6

As you can see when we try to convert `unit_price` to a numerical datatype, we receive a `ValueError` telling us that pandas doesn't understand the number `1.137.99`. This is probably because numbers cannot have multiple decimal points. Let's see if there are any other numbers like this:

> `.` is a wildcard in regex, we need the `\` as an escape

In [ ]:
# Count the number of decimal points in the unit_price
orderlines_df['unit_price'].str.count(r"\.").value_counts()

,count
unit_price,
1,257814
2,36169


Looks like over 36000 rows in `orderlines` are affected by this problem. Let's work out how much that is as a percentage of our total data.

In [66]:
# Count the rows with more than one `.`
mult_decimal_rows = (orderlines_df['unit_price'].str.count(r"\.")>1).sum()

# Find the percentage of corrupted rows
percent_corrupted = (100 * mult_decimal_rows / orderlines_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in our DataFrame have multiple decimal points in the unit_price")

AttributeError: Can only use .str accessor with string values, not floating

This is a bit of a tricky decision as 12.3% is a significant amount of our data... and we might even end up losing a larger portion of our data than this too. For the moment we will delete the rows as we only have 2 weeks for this project and I'd like some quick, accurate results to show. If we have time at the end, we can come back and investigate this problem further, maybe there's a solution?

Each row of `orderlines` represents a product in an order. For example, if order number 175 contained 3 seperate products, then order 175 would have 3 rows in `orderlines`, one row for each of the products. If 2 of those products have 'normal' prices (14.99, 15.85) and 1 has a price with 2 decimal points (1.137.99), we need to remove the whole order and not just the affected row. If we only remove the row with 2 decimal places then any later analysis about products and prices could be misleading.

We therefore need to find the order numbers associated with the rows that have 2 decimal points, and then remove all the associated rows.

In [12]:
# Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = orderlines_df['unit_price'].str.count(r"\.") > 1

# Apply the boolean mask to the orderlines DataFrame. This way we can find the order_id of all the affected orders.
corrupted_order_ids = orderlines_df.loc[multiple_decimal_mask, "id_order"]

# Keep only the rows that do not have multiple decimal points
orderlines_df = orderlines_df.loc[~orderlines_df['id_order'].isin(corrupted_order_ids)]

In [13]:
orderlines_df.shape[0]

216250

We still have 216250 rows in orderlines to work with. This should be more than enough for our evaluation.

Now that all of the 2 decimal point prices have been removed, let's try again to convert the column `unit_price` to the correct datatype.

In [14]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

It worked perfectly

# Challenge: Clean the `products` DataFrame
Now it's your turn. Use the lessons you learnt above and clean the products DataFrame. You don't have to copy exactly what we did. Think about the consequences of your actions, sometimes it is ok to delete rows, other times you may wish to come up with more creative solutions.

In [15]:
## products.csv
# url = "https://drive.google.com/file/d/1afxwDXfl-7cQ_qLwyDitfcCx3u7WMvkU/view?usp=drive_link"
# path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products = pd.read_csv(r"C:\Users\msnch\OneDrive\Bootcamp\Eniacs_Discount\1_original\products.csv")

In [107]:
# your code here
products_df = products.copy()

### Look for Duplicates

In [84]:
# your code here
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          19326 non-null  str  
 1   name         19326 non-null  str  
 2   desc         19319 non-null  str  
 3   price        19280 non-null  str  
 4   promo_price  19326 non-null  str  
 5   in_stock     19326 non-null  int64
 6   type         19276 non-null  str  
dtypes: int64(1), str(6)
memory usage: 4.0 MB


### Look for Missing values


In [108]:
## your code here
products_df = products_df.dropna(axis=0) #This drops all missing values, because thee o takes the 'y' axis?
products_df.info()

<class 'pandas.DataFrame'>
Index: 19223 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          19223 non-null  str  
 1   name         19223 non-null  str  
 2   desc         19223 non-null  str  
 3   price        19223 non-null  str  
 4   promo_price  19223 non-null  str  
 5   in_stock     19223 non-null  int64
 6   type         19223 non-null  str  
dtypes: int64(1), str(6)
memory usage: 4.1 MB


In [43]:
## your code here
products_df['type'].unique()

<ArrowStringArray>
[     '8696',  '13855401',      '1387',      '1230',      '1364',      '1325',
      '5384',      '1334',  '13005399',  '13835403',
 ...
 '113291716', '113281716', '113271716', '113464259',  '5,45E+15', '113851714',
     '11859', '118692158',  '51912158',     '12282']
Length: 126, dtype: str

### Check / Change Data types

In [110]:
# your code here
products_df['price'].str.count(r"\.").value_counts()

# Count the rows with more than one `.`
# mult_decimal_rows = (products_df['price'].str.count(r"\.")>1).sum()

# Find the percentage of corrupted rows
# percent_corrupted = (100 * mult_decimal_rows / products_df.shape[0])

# Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = products_df['price'].str.count(r"\.") > 1

# Remove rows with problematic values in the price column
products_df = products_df.loc[~multiple_decimal_mask]
# products_df.info()

## products_df.loc[multiple_decimal_mask, 'price'] = np.nan # This is code to keep the products, and just remove the problematic price values.


In [87]:
# your code here
products_df['price'] = pd.to_numeric(products_df['price'])
products_df.info()

<class 'pandas.DataFrame'>
Index: 18796 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku          18796 non-null  str    
 1   name         18796 non-null  str    
 2   desc         18796 non-null  str    
 3   price        18796 non-null  float64
 4   promo_price  18796 non-null  str    
 5   in_stock     18796 non-null  int64  
 6   type         18796 non-null  str    
dtypes: float64(1), int64(1), str(5)
memory usage: 3.9 MB


In [73]:
# your code here
products_df['promo_price'].str.count(r"\.").value_counts()

# Count the rows with more than one `.`
mult_decimal_rows = (products_df['promo_price'].str.count(r"\.")>1).sum()

# Find the percentage of corrupted rows
percent_corrupted = (100 * mult_decimal_rows / products_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in our DataFrame have multiple decimal points in the unit_price")
print(f"We will lose a big part of the data, so we cannot remove the promo price from the data set since")

66.69% of the rows in our DataFrame have multiple decimal points in the unit_price
We will lose a big part of the data, so we cannot remove the promo price from the data set since


In [68]:
# your code here
products_df['promo_price'].unique()
products_df['promo_price'].str.count(r"\.").value_counts()

promo_price
2    12535
1     6011
0      250
Name: count, dtype: int64

### 5.1 Further data cleaning - Products

In [105]:
products_df.drop_duplicates(inplace=True)
len(products_df)

10104

In [89]:
products_df['sku'].unique # Checking unique value in this column, seems right!

<bound method Series.unique of 0        RAI0007
1        APP0023
2        APP0025
3        APP0072
4        KIN0007
          ...   
19321    BEL0376
19322    THU0060
19323    THU0061
19324    THU0062
19325    THU0063
Name: sku, Length: 10104, dtype: str>

Before removing duplicates we have **18,796** rows. After removing the duplicates we have **10,104**. We still have enough value to work with, but we lost almost half of the values (meaning we have too many duplicates).

In [95]:
products_df['promo_price'].str.count(r"\.").value_counts()
mult_decimal_rows = (products_df['promo_price'].str.count(r"\.")>1).sum()
print(mult_decimal_rows)

4284


In [98]:
wrong_prices = products_df[
    products_df["promo_price"].str.count(r"\.") > 1
    ]["promo_price"]
print(wrong_prices)

50        1.119.976
51       10.449.923
66        1.199.957
67        1.079.961
97        1.649.896
            ...    
19300     1.199.897
19317     1.599.898
19318     1.399.897
19319     1.699.905
19320     1.499.795
Name: promo_price, Length: 4284, dtype: str


In [103]:
products_df[
    products_df["promo_price"].str.count(r"\.") > 1
]

,sku,name,desc,price,promo_price,in_stock,type
50,APP0367,Apple Mini DisplayPort to DVI Adapter Mac dual...,Adapter Mini Display Port to DVI dual channel ...,119.00,1.119.976,0,1325
51,APP0344,"Apple Thunderbolt Display 27 ""Monitor Mac",Monitor Display 27-inch Apple Thunderbolt (MC9...,1149.00,10.449.923,0,1296
66,MAK0008,Maclocks theft case iPad 2 3 and 4 black with ...,Case antitheft iPad 2 3 and 4 polycarbonate ro...,120.00,1.199.957,0,12635403
67,MAK0007,Maclocks theft case iPad 2 3 and 4 transparent...,Case antitheft iPad 2 3 and 4 polycarbonate ro...,120.00,1.079.961,0,12635403
97,MAK0014,Maclocks safety housing Kiosk iPad 2 3 and 4 b...,Holder and housing iPad 2 3 and 4 aluminum and...,164.99,1.649.896,0,1216
...,...,...,...,...,...,...,...
19300,REP0412,Rear Camera Repair iPhone 7 Plus,It is including parts and labor for iPhone 7 Plus,119.99,1.199.897,0,"1,44E+11"
19317,REP0403,iPad LCD screen repair,Repair service including parts and labor for iPad,159.99,1.599.898,0,"1,44E+11"
19318,REP0402,iPad touch screen repair,Repair service including parts and labor for iPad,139.99,1.399.897,0,"1,44E+11"
19319,KNO0032,"Knomo MacBook Pro Beauchamp Backpack 14 ""Black",Backpack thin nylon mesh internal compartment ...,179.00,1.699.905,1,1392


In [102]:
products_df.head()

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364
